# 06 (PH) - PyCaret comparison: which model family, by numbers

**Who worked on this notebook**

| Who | What |
| --- | --- |
| Peter Henskens | Keeps the split, the metrics and the inputs of 05; decided that this notebook only picks the family (training on all rows is for 07, on AWS for 10), the station sample, the excluded families and the station key as a category; runs the notebook and reviews the leaderboard |
| Claude Code (AI assistant) | Drafted the notebook, the PyCaret settings, the bounded forests, the leaderboard cleaning and the checks; found the LightGBM crash and the MemoryError of the appendix. Logged in `genai-log.md` |

**Goal.** 05 set the bar: the best rule per horizon and a ridge floor. This notebook lets **PyCaret compare the
model families** on the same feature table, the same split and the same scoring, and names per horizon the
family that 07 trains on all rows, 08 tunes and 10 trains on SageMaker. Nothing is decided in advance: the
ranking comes from the leaderboard. It is an analysis step, so it runs on a **station sample** and trains
nothing on the full data.

**Products**

| Product | What it is |
| --- | --- |
| leaderboard per horizon | every family's MAE and RMSE in 3 time-ordered folds inside the training years, on a 10 % station sample |
| validation table per horizon | the top 3 scored on the sample's validation rows with `score_table()`, next to the rules of 05 on the same rows |
| the choice per horizon | the family 07 - 10 continue with, and why (gain over the rules, training time) |
| `models/citibike/06_best_<horizon>_sample.pkl` | the best family as fitted on the sample, a reference for 07, not a model to ship |

**How it is built**
0. Setup (kernel "Python 3.11 (pycaret)", see `requirements_oldpy.txt`).
1. What PyCaret decides here, and what it does not.
2. Load, sort, sample.
3. Horizon day: compare, then score the top 3 on the sample's validation rows.
4. Horizon rest of month: the same, on the rate per day.
5. The choice per horizon.
6. Reading and what carries forward.
Appendix: the LightGBM crash, the MemoryError of the first run, the failures to expect.

**Status: written for review.** The first version ran on 7 October up to the refit on all rows (appendix);
this version stops at the sample on purpose. The reading in part 6 is written after the run, from its outputs.

## 0. Setup

This notebook runs in its own environment: PyCaret 3.3.2 needs Python 3.9 - 3.11 and pins pandas below 2.2 and
numpy below 1.27, so `requirements_oldpy.txt` installs it in `.venv-pycaret` (kernel "Python 3.11 (pycaret)").
It also pins `lightgbm` below 4.6: 4.7.0 crashes on Windows in this environment (appendix).

In [1]:
import sys
import time

import lightgbm
import numpy as np
import pandas as pd
import pycaret
from pycaret.regression import RegressionExperiment
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor

sys.path.insert(0, "..")   # citibike/ -> connection.py and modelling.py
from connection import fmt, notebook_style
from modelling import (load_features, scorable, score_table, rule_predictions, design, TARGETS, MIN_HISTORY_DAYS,
                       FEATURES_DAY, FEATURES_REST, CALENDAR, MODELS_DIR)

notebook_style()
assert sys.version_info[:2] == (3, 11), "pick the kernel 'Python 3.11 (pycaret)' (see requirements_oldpy.txt)"
print(f"python {sys.version.split()[0]}, pycaret {pycaret.__version__}, pandas {pd.__version__}, lightgbm {lightgbm.__version__}")

SEED = 42
SAMPLE_SHARE = 0.10          # share of the stations in the sample (all their rows, every season)
FOLDS = 3                    # time-ordered folds inside the training years
TOP_N = 3                    # families scored on the sample's validation rows
EXCLUDE = ["knn", "svm", "mlp", "kr", "tr", "ard", "ransac"]   # see part 1 for the reason per family
MIN_LEAF = 100               # leaf size of random forest and extra trees (part 1)
CATEGORICAL = ["station_key"] + CALENDAR

python 3.11.0, pycaret 3.3.2, pandas 2.1.4, lightgbm 4.5.0


## 1. What PyCaret decides here, and what it does not

**Fixed before PyCaret starts** (03 §6.3, 05):
- **The split.** PyCaret's default `setup()` draws a random 70/30 split; for a time series that trains on the
  future and flatters every model. So the training years go in as `data`, the validation year 2025 as
  `test_data` (PyCaret calls the holdout "test"; our real test period, 2026, is never passed to PyCaret) and
  the cross-validation inside the training years is `fold_strategy="timeseries"`: 3 folds that each train on
  the earlier rows and score on the next block, with the rows sorted by day so that a block is a period.
- **The inputs.** The same `design()` as 05 §7 for each horizon, plus the **station key as a category** (the
  6 October decision): with 2.455 levels PyCaret target-encodes it (one column: the station's mean target on the
  training rows), the weekday and the month are one-hot as in 05.
- **The scoring.** PyCaret ranks on **MAE**; the validation table per horizon is `score_table()` on the sample's
  validation rows (MAE, RMSE, WAPE, bias), the same function as 05, with the rules on the same rows.

**Decided here**
- **A station sample, and nothing on all rows.** The comparison runs on **10 % of the stations, all their
  rows** (every season, every weekday, quiet and busy stations): about 250.000 training rows and 80.000
  validation rows. That is enough to rank families; training the chosen family on all 2,5 M rows is 07's job
  (and 10's, on SageMaker), where it is done once, with the time and memory it needs.
- **Excluded families**, each for a reason: `knn` (predicts by distance: minutes per prediction and no sense of
  a time series), `svm` and `kr` (kernel methods, quadratic in the rows), `mlp` (a neural network with default
  settings tells nothing here), `tr` and `ransac` (robust fits for outliers; our outliers are storms, not
  errors), `ard` (slow Bayesian ridge, `br` covers it). Everything else is in, including `dummy` (the mean) as
  a sanity floor.
- **Bounded forests.** PyCaret's random forest and extra trees grow every tree until the leaves are pure: on
  257.000 rows that is already gigabytes per model, and the first run died on it (appendix). Both enter the
  comparison as their own estimators with `min_samples_leaf = 100`; a leaf of 100 station-days is also a sane
  regulariser for a count that is noisy at the station level.
- **A failed family is dropped, not ranked.** When a family fails in a fold, PyCaret still prints a row with a
  0 where the score should be, and the first run ranked such a row first. The leaderboard is cleaned (a MAE of
  0 or missing) before the top 3 are chosen; the dropped rows are printed.
- **Rest of month as a rate**, as in 05: the target is departures per remaining day, the prediction is
  multiplied back by the days left before scoring.

In [2]:
df = load_features()
df = df.sort_values(["day", "station_key"]).reset_index(drop=True)    # time order: a fold block is a period
rng = np.random.default_rng(SEED)
stations = np.sort(df.station_key.unique())
sample_keys = set(rng.choice(stations, size=int(len(stations) * SAMPLE_SHARE), replace=False))
df["in_sample"] = df.station_key.isin(sample_keys)

def rows(horizon: str, split: str) -> pd.DataFrame:
    return df[(df.split == split) & scorable(df, horizon) & df.in_sample]

overview = pd.DataFrame({
    "stations": {"all": len(stations), "sample": len(sample_keys)},
    "train rows, day": {"all": int(((df.split == "train") & scorable(df, "day")).sum()), "sample": len(rows("day", "train"))},
    "valid rows, day": {"all": int(((df.split == "valid") & scorable(df, "day")).sum()), "sample": len(rows("day", "valid"))},
    "share of departures, train": {"all": 1.0, "sample": rows("day", "train").y_day.sum() / df[(df.split == "train") & scorable(df, "day")].y_day.sum()},
})
display(overview)


def inputs(d: pd.DataFrame, features: list[str], target: pd.Series) -> pd.DataFrame:
    """The PyCaret frame for the rows of d: the design matrix of 05, the station key as text, the target as y."""
    return (design(d)[features]
            .assign(station_key=d.station_key.astype(str).to_numpy(), y=target.to_numpy(dtype="float64"))
            .reset_index(drop=True))


def experiment(train: pd.DataFrame, valid: pd.DataFrame) -> RegressionExperiment:
    exp = RegressionExperiment()
    exp.setup(data=train, target="y", test_data=valid, index=False,
              fold_strategy="timeseries", fold=FOLDS, data_split_shuffle=False, fold_shuffle=False,
              categorical_features=CATEGORICAL, session_id=SEED, n_jobs=-1, verbose=False, html=False)
    return exp


def families(exp: RegressionExperiment) -> list:
    """Every family of the registry minus EXCLUDE, with the two forests replaced by their bounded versions."""
    ids = [m for m in exp.models().index if m not in EXCLUDE + ["rf", "et"]]
    return ids + [RandomForestRegressor(min_samples_leaf=MIN_LEAF, n_jobs=-1, random_state=SEED),
                  ExtraTreesRegressor(min_samples_leaf=MIN_LEAF, n_jobs=-1, random_state=SEED)]


def compare(exp: RegressionExperiment) -> tuple[pd.DataFrame, list]:
    """compare_models on every family; the leaderboard cleaned of failed rows; the top TOP_N models in its order."""
    include = families(exp)
    models = exp.compare_models(sort="MAE", include=include, n_select=len(include), verbose=False)
    board = exp.pull()
    by_row = dict(zip(board.index, models))                 # compare_models returns the models in leaderboard order
    failed = board[(board.MAE <= 0) | board.MAE.isna()]
    if len(failed):
        print(f"dropped, failed in at least one fold: {', '.join(failed.Model)}")
    board = board.drop(failed.index)
    top = [by_row[i] for i in board.index[:TOP_N]]
    board.index = pd.RangeIndex(1, len(board) + 1, name="rank")   # with custom estimators PyCaret numbers the rows
    return board, top


def predictions(exp: RegressionExperiment, model, valid_inputs: pd.DataFrame, index: pd.Index) -> pd.Series:
    """predict_model() returns a fresh index; put the predictions back on the rows they belong to."""
    out = exp.predict_model(model, data=valid_inputs.drop(columns="y"), verbose=False)
    return pd.Series(out.prediction_label.to_numpy(), index=index).clip(lower=0)

3.872.821 rows from data\citibike\station_day_features.parquet


,stations,"train rows, day","valid rows, day","share of departures, train"
all,2.455,2.480.425,779.151,"1,000000"
sample,245,257.448,79.662,"0,115586"


## 3. Horizon day

### 3.1 Compare the families
`compare_models` fits every family of part 1 in 3 time-ordered folds and ranks on MAE. The leaderboard shows
the mean over the folds; `TT (Sec)` is the training time per fit on the sample, which matters for 07 - 10 and
for the API (the first run, 7 October, took 9,7 minutes for this cell).

In [3]:
train_day, valid_day = rows("day", "train"), rows("day", "valid")
t0 = time.time()
exp_day = experiment(inputs(train_day, FEATURES_DAY, train_day.y_day), inputs(valid_day, FEATURES_DAY, valid_day.y_day))
leaderboard_day, top_day = compare(exp_day)
print(f"horizon day: {fmt(len(train_day))} sample rows, {len(leaderboard_day)} families compared in {fmt((time.time() - t0) / 60, '.1f')} min")
leaderboard_day[["Model", "MAE", "RMSE", "R2", "TT (Sec)"]].round(2)

horizon day: 257.448 sample rows, 18 families compared in 10,4 min


,Model,MAE,RMSE,R2,TT (Sec)
rank,,,,,
1,Light Gradient Boosting Machine,"11,83","20,92","0,92","1,69"
2,Random Forest Regressor,"11,95","22,43","0,91","12,92"
3,Gradient Boosting Regressor,"12,26","21,79","0,91","27,75"
4,Extra Trees Regressor,"12,29","23,04","0,90","7,09"
5,Extreme Gradient Boosting,"13,00","22,95","0,90","2,11"
6,Huber Regressor,"13,14","23,94","0,89","4,01"
7,Lasso Regression,"13,45","23,70","0,90","6,84"
8,Elastic Net,"13,45","23,69","0,90","5,96"
9,Lasso Least Angle Regression,"13,45","23,70","0,90","2,26"


### 3.2 The top 3 on the sample's validation rows
The models that `compare_models` returns are fitted on the whole training part of the sample. Scored on the
sample's validation year with `score_table()`, next to the rules of 05 on exactly the same rows, so the gain
over the rules is read on one table. 05's numbers on **all** validation rows, for scale: 4-week mean 14,57,
ridge 11,92.

In [4]:
valid_inputs = inputs(valid_day, FEATURES_DAY, valid_day.y_day)
preds_day = {f"pycaret: {name}": predictions(exp_day, model, valid_inputs, valid_day.index)
             for name, model in zip(leaderboard_day.Model.head(TOP_N), top_day)}
print(f"Validation 2025, sample ({fmt(len(valid_day))} station-days), horizon day: the rules of 05 and the top 3")
table_day = score_table(valid_day, {**rule_predictions(valid_day, "day"), **preds_day}, "y_day")
table_day.round(2)

Validation 2025, sample (79.662 station-days), horizon day: the rules of 05 and the top 3


,MAE,RMSE,WAPE_%,bias,n
method,,,,,
naive (yesterday),"16,38","31,28","25,13","0,04",79.662
seasonal naive (same weekday last week),"17,58","33,48","26,96","0,19",79.662
"mean of the same weekday, 4 weeks","15,82","29,87","24,26","0,24",79.662
pycaret: Light Gradient Boosting Machine,"11,35","20,63","17,41","1,68",79.662
pycaret: Random Forest Regressor,"11,51","21,88","17,66","-0,77",79.662
pycaret: Gradient Boosting Regressor,"11,60","21,27","17,79","0,11",79.662


## 4. Horizon rest of month

The same two steps on the rate target (`y_rest_of_month / days_in_horizon`), with the inputs of 05 §7 for this
horizon: no weather, the pace and year-earlier rates instead. The leaderboard is on the rate scale; the
validation table multiplies back by the days left and scores the real total, as 05 did. 05's numbers on all
validation rows: weekly pace 149,1, ridge 150,2.

In [5]:
train_rest, valid_rest = rows("rest_of_month", "train"), rows("rest_of_month", "valid")
rate = lambda d: d.y_rest_of_month / d.days_in_horizon
t0 = time.time()
exp_rest = experiment(inputs(train_rest, FEATURES_REST, rate(train_rest)), inputs(valid_rest, FEATURES_REST, rate(valid_rest)))
leaderboard_rest, top_rest = compare(exp_rest)
print(f"horizon rest of month: {fmt(len(train_rest))} sample rows, {len(leaderboard_rest)} families compared in "
      f"{fmt((time.time() - t0) / 60, '.1f')} min (MAE and RMSE in departures per remaining day)")
leaderboard_rest[["Model", "MAE", "RMSE", "R2", "TT (Sec)"]].round(3)

horizon rest of month: 256.959 sample rows, 18 families compared in 11,7 min (MAE and RMSE in departures per remaining day)


,Model,MAE,RMSE,R2,TT (Sec)
rank,,,,,
1,Extra Trees Regressor,"9,366","17,589","0,934","6,760"
2,Light Gradient Boosting Machine,"10,596","19,239","0,920","1,490"
3,Gradient Boosting Regressor,"10,611","19,006","0,922","33,903"
4,Random Forest Regressor,"10,755","19,510","0,918","14,230"
5,Huber Regressor,"10,930","29,696","0,812","4,123"
6,Bayesian Ridge,"11,512","20,808","0,904","3,197"
7,Ridge Regression,"11,515","20,804","0,904","2,083"
8,Linear Regression,"11,515","20,804","0,904","3,757"
9,Extreme Gradient Boosting,"12,172","22,422","0,893","2,010"


In [7]:
valid_inputs = inputs(valid_rest, FEATURES_REST, rate(valid_rest))
preds_rest = {f"pycaret: {name}": predictions(exp_rest, model, valid_inputs, valid_rest.index) * valid_rest.days_in_horizon
              for name, model in zip(leaderboard_rest.Model.head(TOP_N), top_rest)}
print(f"Validation 2025, sample ({fmt(len(valid_rest))} station-days), horizon rest of month: the rules of 05 and the top 3")
table_rest = score_table(valid_rest, {**rule_predictions(valid_rest, "rest_of_month"), **preds_rest}, "y_rest_of_month")
table_rest.round(2)

Validation 2025, sample (79.586 station-days), horizon rest of month: the rules of 05 and the top 3


,MAE,RMSE,WAPE_%,bias,n
method,,,,,
"pace, month to date x days left","183,63","406,41","18,05","27,71",79.586
"pace, last 7 days x days left","160,71","343,64","15,79","17,09",79.586
"last year, same period x growth","169,25","2.687,59","16,63","51,98",79.586
pycaret: Extra Trees Regressor,"118,92","254,28","11,69","-7,34",79.586
pycaret: Light Gradient Boosting Machine,"123,14","248,97","12,10","-31,47",79.586
pycaret: Gradient Boosting Regressor,"130,72","260,69","12,85","12,98",79.586


## 5. The choice per horizon

The family with the lowest validation MAE on the sample, its gain over the best rule on the same rows, and its
training time per fit. The sample-fitted model goes to `models/citibike/` as a reference for 07 (its parameters,
its preprocessing), not as a model to ship: 07 trains the family on all rows.

In [8]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
choice = {}
for horizon, table, board, top, exp in [("day", table_day, leaderboard_day, top_day, exp_day),
                                        ("rest_of_month", table_rest, leaderboard_rest, top_rest, exp_rest)]:
    pycaret_rows = table[table.index.str.startswith("pycaret: ")]
    rule_rows = table[~table.index.str.startswith("pycaret: ")]
    name = pycaret_rows.MAE.idxmin().removeprefix("pycaret: ")
    model = top[list(board.Model.head(TOP_N)).index(name)]
    exp.save_model(model, str(MODELS_DIR / f"06_best_{horizon}_sample"), verbose=False)
    choice[horizon] = {"family": name, "validation MAE": pycaret_rows.MAE.min(), "best rule": rule_rows.MAE.idxmin(),
                       "best rule MAE": rule_rows.MAE.min(), "gain %": (1 - pycaret_rows.MAE.min() / rule_rows.MAE.min()) * 100,
                       "fit time (s)": board.loc[board.Model == name, "TT (Sec)"].iloc[0]}
print(f"saved {', '.join(p.name for p in sorted(MODELS_DIR.glob('06_best_*_sample.pkl')))}")
pd.DataFrame(choice).T

saved 06_best_day_sample.pkl, 06_best_rest_of_month_sample.pkl


,family,validation MAE,best rule,best rule MAE,gain %,fit time (s)
day,Light Gradient Boosting Machine,11.349782,"mean of the same weekday, 4 weeks",15.816970,28.243008,1.693300
rest_of_month,Extra Trees Regressor,118.920658,"pace, last 7 days x days left",160.706917,26.001531,6.760000


## 6. Reading and what carries forward

Run of 7 October 2026 (team lead): 18 families per horizon on 245 stations, 10,4 and 11,7 minutes. Scores on the
sample's validation rows (about 80.000 station-days per horizon).

| Horizon | Best rule | Best family | Gain | Runners-up | Fit time |
| --- | --- | --- | --- | --- | --- |
| day | 4-week mean: MAE 15,82, WAPE 24,3 % | **LightGBM**: 11,35, WAPE 17,4 % | 28 % | random forest 11,51, gradient boosting 11,60 | 1,7 s against 12,9 and 27,8 s |
| rest of month | weekly pace: 160,7, WAPE 15,8 % | **extra trees**: 118,9, WAPE 11,7 % | 26 % | LightGBM 123,1, gradient boosting 130,7 | 6,8 s against 1,5 and 33,9 s |

- **Tree ensembles beat every linear family on both horizons.** The linear families sit at 13,5 on the day folds,
  where 05's ridge was. The gain is largest on the rest of the month, the horizon where 05's ridge could not beat
  the pace: the non-linear effects (season, days left, the year earlier) are what the pace misses.
- **Read the table within itself.** The sample is a little busier than the network (11,6 % of the departures for
  10 % of the stations), so its rules score worse than on all rows (15,82 against 14,57 in 05). The gain over the
  rules is the comparable number, not the MAE.
- **Bias.** LightGBM under-predicts the rest of the month (bias -31 on a mean of about 1.000 departures); extra
  trees is close to unbiased (-7). A lead for 08 if LightGBM is kept for both horizons.
- **One blow-up, no effect.** Least angle regression returned a MAE of 10^10 on the rate target (a numerical
  instability with collinear lags); it ranks last and changes nothing.

**Decision for the team:** two families (the leaderboard winners: LightGBM for the day, extra trees for the rest of
the month) or **LightGBM for both** (3,5 % worse on the rest of the month, one framework for 08, for SageMaker in
10 and for the API, four times faster to fit). Recorded in `genai-log.md` once taken.

**What 07 - 10 take from here:** the family per horizon once decided, the leaderboards as the argument, the
sample-fitted reference models, and the same `score_table()` rows for 09.

## Appendix: the LightGBM crash, the MemoryError of the first run, the failures to expect

**The LightGBM crash** (found in the smoke test of 7 October). With `lightgbm` 4.7.0 in `.venv-pycaret` on
Windows, any LightGBM fit after `import sklearn` (and so after `import pycaret`) ends in `OSError: access
violation reading 0x0` inside `LGBM_DatasetSetField`; inside `compare_models` the family silently disappears
from the leaderboard, because the cross-validation runs in worker processes that import scikit-learn first.
`lightgbm` 4.5.0 works in every import order and with every joblib backend, so `requirements_oldpy.txt` pins
`lightgbm<4.6`; the smoke test asserts that `lightgbm` and `xgboost` are on the leaderboard. Checked by direct
fits with each PyCaret dependency imported first: every package that imports scikit-learn 1.4.2 triggers it,
`numba` and `statsmodels` do not.

**The MemoryError of the first run** (7 October). The first version of this notebook refitted the top 3 on all
2,48 M training rows. Its comparison ran in 9,7 minutes and ranked extra trees first with a MAE of 0,00 next
to an R² of 0,30: the model had failed in at least one fold (an unbounded forest on 257.000 rows is already
gigabytes) and PyCaret printed a 0 for it. The refit of that model on all rows stopped with `MemoryError: could
not allocate 8388608 bytes` inside a tree builder, on a laptop with 16 GB. Three changes: no refit on all rows
in this notebook (07's job), the forests bounded with `MIN_LEAF`, and a leaderboard row with a MAE of 0 or
missing dropped before the top 3 are chosen. The real ranking of that run, for the record: random forest 11,81,
LightGBM 11,83, gradient boosting 12,26, XGBoost 13,00, the linear families 13,1 - 13,7 (MAE on the folds of the
sample, unbounded forests).

**Expected runtime**: about 10 minutes per horizon for the comparison (measured once: 9,7 minutes for the day
horizon), seconds for the rest. Memory: the feature table is 1,2 GB in pandas; the sample experiments are small.

| Symptom | Cause | Fix |
| --- | --- | --- |
| `ModuleNotFoundError: pycaret` or the assertion in part 0 | wrong kernel | pick "Python 3.11 (pycaret)"; create it with the lines at the top of `requirements_oldpy.txt` |
| `lightgbm` missing from a leaderboard, or an access violation | `lightgbm` 4.6 or newer in the environment | `pip install "lightgbm<4.6"` in `.venv-pycaret` (the pin is in `requirements_oldpy.txt`) |
| part 2 pulls from the server instead of the cache | `data/citibike/station_day_features.parquet` is missing: 05 part 5 writes it | run 05 part 5, or let it pull (needs the `.env` login) |
| a row printed as "dropped, failed in at least one fold" | the family ran out of memory or raised in a fold | expected for an unbounded tree model; otherwise rerun with `n_jobs=1` in `experiment()` to see the error |
| a family scores far worse on validation than in the folds | the fold blocks and the validation year differ in season | normal for a 3-fold time split; read the validation table, not the folds, for the choice |